# DataLandia: puentes batch y streaming
Los archivos de salida se generan con `02_puentes_batch_streaming.py`. Los vínculos entre ciudades son simulados; consulte `data/silver/datalandia_v1/README.md`.

In [ ]:
from pathlib import Path
import duckdb

root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent
silver = root / 'data/silver/datalandia_v1'
gold = root / 'data/gold/datalandia_v1'
con = duckdb.connect()
con.execute("SET TimeZone='UTC'")

## Puente batch

In [ ]:
con.execute("SELECT source_system, count(*) asignaciones, count(DISTINCT datalandia_zone_id) zonas FROM read_parquet(?) GROUP BY 1 ORDER BY 1", [str(silver / 'bridge_batch.parquet')]).df()

## Puente de sensores

In [ ]:
con.execute("SELECT * FROM read_parquet(?) ORDER BY datalandia_zone_id", [str(silver / 'bridge_streaming.parquet')]).df()

## Indicadores conjuntos
Las ventanas están en UTC. Los campos de Censo y Uber son referencias históricas, no observaciones de 2024.

In [ ]:
con.execute("SELECT * FROM read_parquet(?) WHERE sensor_observations IS NOT NULL AND trip_count IS NOT NULL ORDER BY window_start_utc, datalandia_zone_id LIMIT 20", [str(gold / 'zone_5min.parquet')]).df()